In [ ]:
# notebook to open all clinvar predictions and filter vcf by chromosome for generating predictions with updated indel handling/including indels
# update to more recent clinvar version: clinvar_20260104.vcf.gz

In [1]:
# import packages
import pandas as pd
from cyvcf2 import VCF
import pybedtools
import seaborn as sns
from collections import Counter

In [2]:
# tabix index the vcf
#!tabix -p vcf ../raw_data/clinvar_20230930.vcf.gz
#!tabix -p vcf ../raw_data/clinvar_20260104.vcf.gz

In [3]:
# open full ClinVar VCF
# iterate through the VCF and reformat to a DF for filtering and saving chrom specific VCFs for generating updated predictions
chrom = []
pos = []
Idee = [] # we'll combine the clinvar ID, CLNSIG, and chr:pos:ref:alt ID
ref = []
alt = []
info = []

for variant in VCF('../raw_data/clinvar_20260104.vcf.gz'):
    if len(variant.ALT) == 0:
        continue
    else:
        chrom.append(f'chr{variant.CHROM}')
        pos.append(variant.POS)
        Idee.append(f'{variant.ID};{variant.INFO.get('CLNSIG')};{(':').join([f'chr{variant.CHROM}', str(variant.POS), variant.REF, variant.ALT[0]])}')
        ref.append(variant.REF)
        alt.append(variant.ALT[0])
        info.append('.')

clinvar_all_df = pd.DataFrame({
    'CHROM' : chrom,
    'POS' : pos,
    'ID' : Idee,
    'REF' : ref,
    'ALT' : alt,
    'INFO' : info
})

In [4]:
# filter variants for:
# 1. only autosomes
clinvar_autosomes = clinvar_all_df[clinvar_all_df['CHROM'].isin([f'chr{i}' for i in range(23)])]
# 2. variants with a max insertion/deletion size of 10bp
clinvar_indel_filter = clinvar_autosomes[(clinvar_autosomes['REF'].str.len() <= 10) &
                                         (clinvar_autosomes['ALT'].str.len() <= 10)]

In [5]:
# convert the filtered VCF to a BED file for filtering for non-coding variants with bedtools
clinvar_BED = pd.DataFrame({
    0 : clinvar_indel_filter['CHROM'],
    1 : clinvar_indel_filter['POS'] - 1,
    2 : clinvar_indel_filter['POS'],
    3 : clinvar_indel_filter['ID']
})
# convert to a pybedtoools object
clinvar_indel_BEDtool = pybedtools.BedTool.from_dataframe(clinvar_BED)

In [6]:
# open exon BED file for filtering exons + splice sites
splice_appended_exon_BED = pybedtools.BedTool('../raw_data/archive2/gencode.v44.protein.coding.exons.splice.autosomes.v2.bed')
# subtract exon overlapping variants
exon_filtered_clinvar_BED = clinvar_indel_BEDtool.intersect(splice_appended_exon_BED, v=True)

In [9]:
# open basic annotation set for comparison
splice_appended_exon_BASIC = pybedtools.BedTool('../raw_data/archive2/gencode.v44.basic.annotation.exons.splice.autosomes.v2.bed')
# subtract exon overlapping variants
exon_filtered_clinvar_BED_basic = clinvar_indel_BEDtool.intersect(splice_appended_exon_BASIC, v=True)

In [10]:
len(exon_filtered_clinvar_BED)

373427

In [11]:
len(exon_filtered_clinvar_BED_basic)

324999

In [19]:
len(exon_filtered_clinvar_BED_basic.to_dataframe())

324999

In [17]:
# check if all basic ids are in protein coding
# len(exon_filtered_clinvar_BED.to_dataframe()[exon_filtered_clinvar_BED.to_dataframe()['name'].isin(exon_filtered_clinvar_BED_basic.to_dataframe()['name'].tolist())])

# they are - you can use that same DF in the analysis but save the intersection as a df for adding a column to final DF
exon_filtered_clinvar_BED_basic.to_dataframe().to_csv(
    '../processed_data/clinvar_exon_filtered_gencode_basic_gff.tsv',
    sep = '\t',
    index = False,
)

In [12]:
# convert the filtered BED file back to a VCF
chrom = []
pos = []
idee = []
ref = []
alt = []
info = []
for i in exon_filtered_clinvar_BED.to_dataframe()['name']:
    # break out the id for getting chrom, pos, ref, alt, etc.
    eyeD = i.split(';')[-1].split(':')
    # chrom
    chrom.append(eyeD[0])
    # pos
    pos.append(eyeD[1])
    # append full 'info' column to the id column
    idee.append(i)
    # ref
    ref.append(eyeD[2])
    # alt
    alt.append(eyeD[-1])
    # info
    info.append('.')
exon_filtered_clinvar_vcf = pd.DataFrame({
    'CHROM' : chrom,
    'POS' : pos,
    'ID' : idee,
    'REF' : ref,
    'ALT' : alt,
    'INFO' : info
})

In [ ]:
# convert the filtered BED file back to a VCF
chrom = []
pos = []
idee = []
ref = []
alt = []
info = []
for i in exon_filtered_clinvar_BED.to_dataframe()['name']:
    # break out the id for getting chrom, pos, ref, alt, etc.
    eyeD = i.split(';')[-1].split(':')
    # chrom
    chrom.append(eyeD[0])
    # pos
    pos.append(eyeD[1])
    # append full 'info' column to the id column
    idee.append(i)
    # ref
    ref.append(eyeD[2])
    # alt
    alt.append(eyeD[-1])
    # info
    info.append('.')
exon_filtered_clinvar_vcf = pd.DataFrame({
    'CHROM' : chrom,
    'POS' : pos,
    'ID' : idee,
    'REF' : ref,
    'ALT' : alt,
    'INFO' : info
})

In [21]:
exon_filtered_clinvar_BED.head()

chr1	66925	66926	3385321;Uncertain_significance;chr1:66926:AG:A
 chr1	139319	139320	4506524;None;chr1:139320:C:G
 chr1	139846	139847	4506682;None;chr1:139847:C:G
 chr1	766398	766399	4286633;None;chr1:766399:GAATA:G
 chr1	778061	778062	4501594;None;chr1:778062:T:C
 chr1	809283	809284	3892489;Benign;chr1:809284:T:TGGTCAATCA
 chr1	818682	818683	4466395;None;chr1:818683:G:T
 chr1	874049	874050	4473043;None;chr1:874050:C:A
 chr1	917749	917750	4474871;None;chr1:917750:G:A
 chr1	917836	917837	4474881;None;chr1:917837:G:A
 

In [22]:
# iterate through the filtered clinvar DF and save chromosome specific tsvs for predictions
for chrom in exon_filtered_clinvar_vcf['CHROM'].unique():
    # filter
    chrom_df = exon_filtered_clinvar_vcf[exon_filtered_clinvar_vcf['CHROM'] == chrom]
    # save to disk
    chrom_df.to_csv(f'../processed_data/chrom_vcfs/{chrom}_clinvar_20260104.tsv', sep = '\t', header = None, index = False)